In [7]:
import re

import pandas as pd
import numpy as np

In [8]:
file_name = '../../data/2018-2025/ncsl_ai_db_2018_2025_clean.csv'
df = pd.read_csv(file_name)

In [9]:
# Map the bill status sub-types to main types - {Passed, Failed, Pending, Vetoed}
bill_status_map = {
    "Enacted": "Passed",
    "Failed": "Failed",
    "Failed - Adjourned": "Failed",
    "Adopted": "Passed",
    "Pending": "Pending",
    "Pending - Carryover": "Pending",
    "To Governor": "Pending",
    "Override Pending": "Pending",
    "Vetoed": "Vetoed"
}
df["status_type"] = df.status.map(bill_status_map)

In [10]:
# Map the full-title states into abbreviations

import us

state_mapping = us.states.mapping('name', 'abbr')
state_mapping.pop("Virgin Islands")
state_mapping['U.S. Virgin Islands'] = 'VI'

df.state = df.state.map(state_mapping)

df['jurisdiction_type'] = df.state.map(
    lambda s: "Territory" if s in {"PR", "VI", "GU", "AS", "MP"}
    else "District" if s == "DC" else "State")

In [11]:
# Map the audience from the bill_number to the session's chamber
bill_audience_map = {'A': 'Assembly Bill',
                     'ACR': 'Assembly Concurrent Resolution',
                     'AJR': 'Assembly Joint Resolution',
                     'AR': 'Assembly Resolution',
                     'B': 'Bill',
                     'BDR': 'Bill Draft Request',
                     'D': 'Draft Bill',
                     'H': 'House Bill',
                     'HCR': 'House Concurrent Resolution',
                     'HD': 'House Docket',
                     'HJM': 'House Joint Memorial',
                     'HJR': 'House Joint Resolution',
                     'HM': 'House Memorial',
                     'HR': 'House Resolution',
                     'HSB': 'House Study Bill ',
                     'L': 'Lesislative Bill',
                     'LCR': 'Legislative Council Resolution',
                     'LR': 'Legislative Reference',
                     'S': 'Senate Bill',
                     'SCR': 'Senate Concurrent Resolution',
                     'SD': 'Senate Docket',
                     'SJR': 'Senate Joint Resolution',
                     'SM': 'Senate Memorial',
                     'SPB': 'Senate Bill',
                     'SR': 'Senate Resolution',
                     'SSB': 'Senate Study Bill',
                     'WS': 'Senate Bill'}

# Split 'bill_number' into chamber prefix and bill number into columns
df[['chamber_prefix', 'bill_number']] = df.bill_number.str.split(
    ' ', n=1, expand=True)

# Get full chamber name from the prefix
df["chamber_name"] = df.chamber_prefix.map(bill_audience_map)

In [12]:
# Add year each session took place and whether it was a special session
session = pd.to_numeric(df.bill_id.str.extract(r"^[A-Z]{2}(\d{4})")[0])
df["session_year"] = session.fillna(df.page_year).astype(int)

df["is_special_session"] = (df.bill_number.str.contains("session", case=False)
                            .groupby(df.bill_id.fillna(df.index.to_series())).transform("any"))

AttributeError: 'DataFrame' object has no attribute 'page_year'

In [ ]:
# Clean bill numbers and create an integer type column
df.bill_number = df.bill_number.str.extract(r"^(\d+)")[0].astype(int)

In [ ]:
document_types = ['Resolution', 'Study Bill',
                  'Memorial', 'Docket', 'Draft', 'Reference']

pattern = f"(?i)({'|'.join(document_types)})"

# Extract the matching document type, will "Bill" as default match
df['document_type'] =  df['chamber_name'].str.extract(pattern, expand=False).fillna('Bill')

In [ ]:
# Types of legislation
df["is_resolution"] = df.chamber_name.str.contains("Resolution")
df["is_joint_resolution"] = df.chamber_name.str.contains("Joint Resolution")
df["is_concurrent_resolution"] = df.chamber_name.str.contains(
    "Concurrent Resolution")
df["is_simple_resolution"] = df.chamber_prefix.isin(["HR", "SR", "AR"])

In [ ]:
# Gets the type of each state/area - {State, Territory, District}
df['jurisdiction_type'] = df.state.map(
    lambda s: "Territory" if s in {"PR", "VI", "GU", "AS", "MP"}
    else "District" if s == "DC" else "State")

In [ ]:
# Get # of companion bills (same bill introduced through different legislative bodies)
df["companion_bill_count"] = df.groupby(
    ["state", "session_year", "title", "summary"], dropna=False).bill_id.transform("size")

In [ ]:
# Fill title/category gaps from the latest earlier listing of the same bill that has a value
for c in ["title", "category"]:
    latest = (df.assign(**{c: df[c].replace("", np.nan)})
                .dropna(subset=[c]).groupby("bill_id")[c].last())
    df[c] = df[c].replace("", np.nan).fillna(df.bill_id.map(latest))

In [ ]:
# Now we can turn the multi-label category list into one-hot-encoded columns
df.category = df.category.fillna("").str.lower()
df.category = df.category.str.replace(' ', '_')

# One hot encoding categories from multi-label text
# Split strings by separator, strip whitespace, and create dummy variables
dummies = df.category.fillna("").str.get_dummies(sep=";")
dummies = dummies.rename(columns=lambda c: "category_" +
                         re.sub(r"\W+", "_", c.lower()))

df["num_categories"] = dummies.sum(axis=1)
df = pd.concat([df, dummies], axis=1)
# df.columns

In [ ]:
# Replace curly quotes in 'title' and 'summary columns'
quotes = str.maketrans({"\u2018": "'", "\u2019": "'",
                       "\u201c": '"', "\u201d": '"'})
for c in ["title", "summary"]:
    df[c] = df[c].map(lambda s: s.translate(quotes)
                      if isinstance(s, str) else s)

In [ ]:
# TODO: Final data quality checks
assert df.bill_id.is_unique
assert (df.session_year <= df.page_year).all()
# assert df.state.notna().all() and df.bill_num.notna().all()
# assert set(df.category.dropna().str.split(";").explode()) <= set(categories)
# assert (df.isnull().sum().sum() == 0)
# print(df.isnull().sum().sort_values(ascending=False))
# column_order = ["bill_id", "state", "session_year", ""]
# assert set(df.columns) == set(column_order), set(df.columns) ^ set(column_order)

df = df.drop(columns=["category", "category_", "chamber_name", "chamber_name", 
                      "page_year"])

# Export final csv to dataset
file_name = '../../data/2018-2025/ncsl_ai_db_2018_2025_clean.csv'
df.to_csv(file_name, index=False)

# View final columns for 2018 - 2025 bills dataset
print(df.columns)

Index(['bill_id', 'state', 'bill_number', 'status', 'title', 'summary',
       'status_type', 'jurisdiction_type', 'chamber_prefix', 'session_year',
       'is_special_session', 'document_type', 'is_resolution',
       'is_joint_resolution', 'is_concurrent_resolution',
       'is_simple_resolution', 'companion_bill_count', 'num_categories',
       'category_appropriations', 'category_audit',
       'category_child_pornography', 'category_criminal_use',
       'category_cybersecurity', 'category_education_training',
       'category_education_use', 'category_effect_on_labor_employment',
       'category_elections', 'category_energy', 'category_government_use',
       'category_health_use', 'category_housing', 'category_impact_assessment',
       'category_judicial_use', 'category_notification',
       'category_oversight_governance', 'category_personhood',
       'category_private_right_of_action', 'category_private_sector',
       'category_provenance', 'category_responsible_use', 'cat